# Exploratory Data Analysis — Deep-Space Optical Chip Thermal Dataset
Author: A Taylor

In [ ]:
# Install dependencies (if needed)
# %pip install pandas matplotlib seaborn datasets plotly

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datasets import load_dataset

print("Imports complete — Author: A Taylor")

In [ ]:
# Load the dataset from HuggingFace
ds = load_dataset("Taylor658/deep-space-optical-chip-thermal-dataset", split="train")
df = ds.to_pandas()

print(f"Dataset shape: {df.shape}")
display(df.head())
df.info()

## Strategy Distribution

In [ ]:
# Pie chart of strategy_type distribution
strategy_counts = df["strategy_type"].value_counts().reset_index()
strategy_counts.columns = ["Strategy", "Count"]

fig = px.pie(
    strategy_counts,
    values="Count",
    names="Strategy",
    title="Strategy Type Distribution",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

## Material Properties Comparison

In [ ]:
# Material properties bar chart
material_props = {
    "Silicon": {"dn_dT": 1.86e-4, "alpha": 2.6e-6},
    "Silicon Nitride": {"dn_dT": 2.45e-5, "alpha": 8.0e-7},
    "Polymer": {"dn_dT": 1.1e-4, "alpha": 2.2e-6},
    "Indium Phosphide": {"dn_dT": 3.4e-4, "alpha": 4.6e-6},
}

prop_data = []
for mat, props in material_props.items():
    prop_data.append({"Material": mat, "Property": "dn/dT", "Value": props["dn_dT"]})
    prop_data.append({"Material": mat, "Property": "\u03b1 (CTE)", "Value": props["alpha"]})

prop_df = pd.DataFrame(prop_data)
fig = px.bar(
    prop_df, x="Material", y="Value", color="Property",
    barmode="group", log_y=True,
    title="dn/dT and Thermal Expansion Coefficient by Material",
)
fig.show()

## Thermal Effects by Environment

In [ ]:
# Heatmap of thermal_effect vs environment_location counts
ct_effect_env = pd.crosstab(df["thermal_effect"], df["environment_location"])

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(ct_effect_env, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
ax.set_title("Thermal Effect vs Environment Location")
plt.tight_layout()
plt.show()

## Instrument × Material Coverage

In [ ]:
# Crosstab heatmap of instrument vs material_name
ct_inst_mat = pd.crosstab(df["instrument"], df["material_name"])

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(ct_inst_mat, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Instrument vs Material Coverage")
plt.tight_layout()
plt.show()

## Strategy by Environment

In [ ]:
# Grouped bar chart of strategy_type breakdown per environment
ct_strat_env = pd.crosstab(df["environment_location"], df["strategy_type"]).reset_index()
ct_melted = ct_strat_env.melt(
    id_vars="environment_location",
    var_name="Strategy",
    value_name="Count",
)

fig = px.bar(
    ct_melted,
    x="environment_location",
    y="Count",
    color="Strategy",
    barmode="group",
    title="Strategy Type Breakdown by Environment",
    labels={"environment_location": "Environment"},
)
fig.show()